## Retailpulse Delta Lake Exploration
Delta Lake Time Travel & ACID Properties Overview.
This notebook demonstrates:

##### 1. DESCRIBE HISTORY
  - View all versions and operations on the Delta table
  - Tracks timestamps, operations, and metrics for each version

In [0]:
display(spark.sql("DESCRIBE HISTORY workspace.retailpulse_bronze.orders"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-05-14T14:21:51.000Z,72254656503404,ihtheram.chowdhury@outlook.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 1e96296c-4ee4-472a-b1f8-4a61dba8bfbf, epochId -> 0, statsOnLoad -> true)",null,List(4398896065216441),9dbea101-8343-438a-b5e9-a35a866293dd,0514-123753-d361md9i-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 500000, numOutputBytes -> 4304108, numAddedFiles -> 1)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-05-14T13:21:20.000Z,72254656503404,ihtheram.chowdhury@outlook.com,CLONE,"Map(source -> workspace.bronze.orders, sourceVersion -> 5, isShallow -> false)",null,List(2845739838953589),7bc0bbce-d277-4b32-a0e4-88f79391b023,0514-123753-d361md9i-v2n,-1,Serializable,false,"Map(removedFilesSize -> 0, numRemovedFiles -> 0, sourceTableSize -> 4304108, numCopiedFiles -> 1, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, copiedFilesSize -> 4304108, sourceNumOfFiles -> 1)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


##### 2. TIME TRAVEL (VERSION AS OF)
  - Query historical versions of the table
  - Use VERSION AS OF clause to access previous snapshots
  - Useful for auditing and debugging data issues

In [0]:
# Query version 0 (before data was loaded)
df_v0 = spark.sql("SELECT * FROM workspace.retailpulse_bronze.orders VERSION AS OF 0")
print(f"Version 0 row count: {df_v0.count():,}")

# Query version 1 (current version)
df_v1 = spark.sql("SELECT * FROM workspace.retailpulse_bronze.orders VERSION AS OF 1")
print(f"Version 1 row count: {df_v1.count():,}")

Version 0 row count: 500,000
Version 1 row count: 1,000,000


##### 3. RESTORE
  - Rollback table to a previous version
  - Quickly recover from bad writes or data quality issues
  - Restore is itself a tracked operation in history

In [0]:
# Append duplicate data (bad write)
spark.sql("""
    INSERT INTO workspace.retailpulse_bronze.orders
    SELECT * FROM workspace.retailpulse_bronze.orders
""")

# Check row count after bad write
count = spark.sql("SELECT COUNT(*) FROM workspace.retailpulse_bronze.orders").collect()[0][0]
print(f"Row count after bad write: {count:,}")

Row count after bad write: 2,000,000


In [0]:
# RESTORE to Version 1 (before the bad write)
spark.sql("RESTORE TABLE workspace.retailpulse_bronze.orders TO VERSION AS OF 1")

# Verify the restore worked
count = spark.sql("SELECT COUNT(*) FROM workspace.retailpulse_bronze.orders").collect()[0][0]
print(f"Row count after RESTORE: {count:,}")

Row count after RESTORE: 1,000,000


In [0]:
# View updated history
spark.sql("DESCRIBE HISTORY workspace.retailpulse_bronze.orders").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=30)

+-------+-------------------+----------------+------------------------------+
|version|          timestamp|       operation|              operationMetrics|
+-------+-------------------+----------------+------------------------------+
|      3|2026-05-14 15:08:09|         RESTORE|{numRestoredFiles -> 0, rem...|
|      2|2026-05-14 14:41:45|           WRITE|{numFiles -> 1, numOutputRo...|
|      1|2026-05-14 14:21:51|STREAMING UPDATE|{numRemovedFiles -> 0, numO...|
|      0|2026-05-14 13:21:20|           CLONE|{removedFilesSize -> 0, num...|
+-------+-------------------+----------------+------------------------------+




##### 4. ACID Properties
  - Atomicity: All-or-nothing writes ensure no partial/corrupted states
  - Consistency: Schema validation on every write
  - Isolation: Concurrent operations see consistent snapshots
  - Durability: Committed data survives failures

In [0]:
spark.sql("""
    DESCRIBE DETAIL workspace.retailpulse_bronze.orders
""").select("format", "numFiles", "sizeInBytes", "minReaderVersion", "minWriterVersion").show(truncate=False)

print("""
ACID Guarantees in Delta Lake:
- Atomicity:   Each write either fully succeeds or fully fails — no partial writes
- Consistency: Table is always in a valid state (schema enforced)
- Isolation:   Concurrent readers see a consistent snapshot while writes happen
- Durability:  Once committed, data is permanent even if the cluster crashes
""")

+------+--------+-----------+----------------+----------------+
|format|numFiles|sizeInBytes|minReaderVersion|minWriterVersion|
+------+--------+-----------+----------------+----------------+
|delta |2       |8608216    |3               |7               |
+------+--------+-----------+----------------+----------------+


ACID Guarantees in Delta Lake:
- Atomicity:   Each write either fully succeeds or fully fails — no partial writes
- Consistency: Table is always in a valid state (schema enforced)
- Isolation:   Concurrent readers see a consistent snapshot while writes happen
- Durability:  Once committed, data is permanent even if the cluster crashes




Delta Lake enables these capabilities through:
- Transaction Log: Tracks all changes in JSON format
- Versioning: Each operation creates a new version
- MVCC: Multi-version concurrency control for isolation
""")